In [8]:
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from time import sleep
import json
import os
client = MongoClient('mongodb://localhost:27017/')
client.drop_database('simplize')
db = client['simplize']
driver = webdriver.Chrome()

url = 'https://simplize.vn/co-phieu/nganh/tai-chinh'
driver.get(url)
sleep(2)
linh_vuc = driver.find_elements(By.XPATH, "//div[contains(@class,'simplize-row css-pmt33i')]")
collection_list = []

for chay_linh_vuc in linh_vuc:
    data = chay_linh_vuc.text
    collection_name =data
    collection = db[collection_name]
    collection.insert_one({"linhvuc": data})
    collection_list.append(data)

print(collection_list)


['Tài chính ngân hàng', 'Chứng khoán và Ngân hàng đầu tư', 'Bảo hiểm']


In [9]:
len(linh_vuc)

3

In [10]:
if linh_vuc:
    linh_vuc[0].click()  
sleep(2)

In [11]:
# Lấy toàn bộ thông tin theo XPath
cac_ma = driver.find_elements(By.XPATH, "//div[contains(@class,'css-70qvj9')]")
ma_list = []  # Tạo list để chứa các thông tin vừa lấy được

for ma in cac_ma:
    ma_list.append(ma.text)  # Thêm thông tin vào list



In [12]:
# Lấy toàn bộ thông tin theo XPath
cac_ma = driver.find_elements(By.XPATH, "//div[contains(@class,'css-70qvj9')]")
ma_list = []  # Tạo list để chứa các thông tin vừa lấy được

for ma in cac_ma:
    ma_list.append(ma.text)  # Thêm thông tin vào list



In [13]:
ma_list

['VCB',
 'BID',
 'CTG',
 'TCB',
 'VPB',
 'MBB',
 'ACB',
 'LPB',
 'HDB',
 'STB',
 'VIB',
 'SSB',
 'TPB',
 'SHB',
 'EIB',
 'MSB',
 'OCB',
 'NAB',
 'BAB',
 'EVF',
 'ABB',
 'PGB',
 'BVB',
 'VBB',
 'VAB',
 'NVB',
 'KLB',
 'SGB',
 'TIN']

# cắt bớt list để chạy nhanh :))

In [14]:
to_remove=('CTG', 'TCB', 'VPB', 'MBB', 'ACB', 'LPB', 'HDB', 'STB', 'VIB', 'SSB', 'TPB', 'SHB', 'EIB', 'MSB', 'OCB', 'NAB', 'BAB', 'EVF', 'ABB', 'PGB', 'BVB', 'VBB', 'VAB', 'NVB', 'KLB', 'SGB')
ma_list = [x for x in ma_list if x not in to_remove]
print(ma_list)


['VCB', 'BID', 'TIN']


# bỏ đoạn trên để lấy full list

In [15]:
# Truy cập vào từng liên kết theo định dạng
for ma in ma_list:
    lich_su_gia_url = f"https://simplize.vn/co-phieu/{ma}/lich-su-gia"  # Tạo URL từ từng giá trị trong ma_list
    driver.get(lich_su_gia_url)  # Truy cập vào từng liên kết
    sleep(2)
    # Lấy toàn bộ thông tin giá
    ls_gia = driver.find_elements(By.XPATH, "//tr[contains(@class,'simplize-table-row simplize-table-row-level-0')]")
    ls_gia_list = []  # Tạo list để chứa thông tin giá

    for row in ls_gia:
        row_data = {
            "date": row.find_element(By.XPATH, ".//td[1]").text,              # Ngày
            "opening_price": row.find_element(By.XPATH, ".//td[2]").text,     # Giá mở cửa
            "highest_price": row.find_element(By.XPATH, ".//td[3]").text,     # Giá cao nhất
            "lowest_price": row.find_element(By.XPATH, ".//td[4]").text,      # Giá thấp nhất
            "closing_price": row.find_element(By.XPATH, ".//td[5]").text,     # Giá đóng cửa
            "price_change": row.find_element(By.XPATH, ".//td[6]").text,       # Thay đổi giá
            "percent_change": row.find_element(By.XPATH, ".//td[7]").text,     # % thay đổi
            "volume": row.find_element(By.XPATH, ".//td[8]").text,             # Khối lượng
        }
        ls_gia_list.append(row_data)  # Thêm dữ liệu của hàng vào danh sách
    with open(f"{ma}_lich_su_gia.json", "w", encoding='utf-8') as json_file:
        json.dump(ls_gia_list, json_file, ensure_ascii=False, indent=4)  # Lưu danh sách vào file JSON
    # Trở về trang trước để nhấp vào mã tiếp theo
    driver.back()

collection_name = collection_list[0]  # Chọn collection đầu tiên
collection = db[collection_name]  # Truy cập collection đầu tiên

for ma in ma_list:
    json_file_path = f"{ma}_lich_su_gia.json"
    
    # Kiểm tra xem file có tồn tại không
    if os.path.exists(json_file_path):
        with open(json_file_path, "r", encoding='utf-8') as json_file:
            ls_gia_list = json.load(json_file)  # Đọc dữ liệu từ file JSON

            # Cập nhật object của mã cổ phiếu
            collection.update_one(
                {"linhvuc": collection_name},  # Tìm đối tượng có giá trị là "Tài chính ngân hàng"
                {"$set": {f"lich_su_gia.{ma}": ls_gia_list}},  # Thêm lịch sử giá vào trường con theo mã cổ phiếu
                upsert=True  # Tạo mới nếu không tồn tại
            )

In [16]:
driver.quit()